# Data Science 1 · Linear and logistic regression

**Lesson:** Data Science 1, Day 1 · **Time:** about 15 minutes · **ISLP:** 3.1–3.3, 4.1–4.3 · **Data:** CADets release 1

**What you will do**
- Predict length of stay: first with the mean, then with a line by age decade
- Split one patient's stay into the mean, what the line explains and the error; then do it for 50 patients and get R²
- Go from the odds ratio of a 2×2 table to a logistic regression, and calculate a patient's risk of death

We follow one patient from the slides all the way: **P00503**, age 84, admitted with an infection, who stayed 11.0 days.

Run each cell in order with **Shift + Enter**. Cells marked **Your turn** have a blank (`____`) to fill in.

In [ ]:
# Setup: load the CADets course dataset. Run this cell first.
# RELEASE: 1 from Day 1, 2 from Day 2, 3 from Day 5. Use the release participants have on your lesson day.
RELEASE = 1

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

FILES = {1: "cadets_admissions_release1_core.csv",
         2: "cadets_admissions_release2_intervention.csv",
         3: "cadets_admissions_release3_full.csv"}
DATA_URL = "https://raw.githubusercontent.com/Rotem-Gershon/cadets-notebooks/main/data/"

def load_cadets(release=RELEASE):
    name = FILES[release]
    if os.path.exists(name):  # a copy uploaded to this session
        return pd.read_csv(name)
    try:  # the course repository, from the release day on
        return pd.read_csv(DATA_URL + name)
    except Exception:
        pass
    try:  # before the release day: upload the file from the course website's dataset page
        from google.colab import files
        print(f"Release {release} is not published yet. Upload {name} from the course website's dataset page.")
        files.upload()
        return pd.read_csv(name)
    except ImportError:
        raise FileNotFoundError(f"Put {name} in the same folder as this notebook.")

df = load_cadets()
print(f"Release {RELEASE}: {df.shape[0]:,} admissions, {df.shape[1]} columns")

## Step 1 · From the mean to a line

Without a model, the best guess for every patient is the mean length of stay. Then we look at age, in decades (8 = age 80 to 89).

In [ ]:
import statsmodels.formula.api as smf

df["age_decade"] = df["age"] // 10
print("Mean length of stay:", round(df["los_days"].mean(), 2), "days")
patient_503 = df[df["patient_id"] == "P00503"]
print(patient_503[["patient_id", "age", "age_decade", "heart_failure", "admit_diagnosis_group", "systolic_bp", "los_days"]])
df.groupby("age_decade")["los_days"].agg(["count", "mean"]).round(2)

The mean stay rises with age. `smf.ols` fits the straight line closest to all the patients (least squares).

In [ ]:
model_age = smf.ols("los_days ~ age_decade", data=df).fit()
print(model_age.params.round(3))

plt.scatter(df["age_decade"], df["los_days"], s=6, alpha=0.2)
plt.scatter(8, 11.0, s=80, color="black", label="P00503")
plt.axhline(df["los_days"].mean(), ls="--", color="grey", label="Mean")
decades = pd.DataFrame({"age_decade": [0, 10]})
plt.plot(decades["age_decade"], model_age.predict(decades), color="red", label="Regression line")
plt.xlabel("Age decade"); plt.ylabel("Length of stay (days)"); plt.legend(); plt.show()

- **Intercept (1.08):** the predicted stay at age 0, where the line meets the vertical axis.
- **Slope (0.70):** each additional decade of age adds 0.7 days.

### Your turn 1

Use the equation to predict the stay of patient P00503, age 84: replace the blank with the age decade.

In [ ]:
patient = pd.DataFrame({"age_decade": [____]})
print("Predicted stay:", round(model_age.predict(patient)[0], 2), "days")

## Step 2 · How much does the line explain? R²

First one patient. P00503's stay = the mean + what the line adds + the error.

In [ ]:
mean_los = df["los_days"].mean()
line_503 = model_age.predict(patient_503).iloc[0]
actual_503 = patient_503["los_days"].iloc[0]
print(f"Mean {mean_los:.2f}   line {line_503:.2f}   actual {actual_503:.2f}")
print(f"Explained by the line = line - mean   = {line_503 - mean_los:.2f}")
print(f"Error                 = actual - line = {actual_503 - line_503:.2f}")

Now the same split for the 50 patients on the slides. Square each part and add them up.

In [ ]:
ids_50 = ['P01680', 'P01868', 'P01617', 'P00497', 'P01515', 'P01679', 'P00488', 'P01257', 'P00486', 'P01801', 'P00473', 'P00316', 'P01995', 'P00045', 'P00075', 'P00614', 'P00901', 'P00851', 'P00224', 'P01804', 'P01148', 'P01020', 'P01045', 'P01528', 'P01158', 'P01042', 'P00344', 'P00019', 'P00984', 'P01133', 'P01877', 'P01908', 'P01595', 'P00256', 'P00990', 'P00541', 'P01202', 'P01284', 'P00177', 'P01968', 'P01829', 'P01155', 'P00670', 'P01521', 'P00872', 'P00395', 'P00562', 'P00340', 'P00915', 'P00503']
sample_50 = df[df["patient_id"].isin(ids_50)]
line_50 = model_age.predict(sample_50)

explained = ((line_50 - mean_los) ** 2).sum()
error = ((sample_50["los_days"] - line_50) ** 2).sum()
total = ((sample_50["los_days"] - mean_los) ** 2).sum()
print(f"Explained {explained:.0f} + error {error:.0f} = total {total:.0f}")
print("R² = explained / total =", round(explained / total, 3))
print("R² for all 2,000 patients:", round(model_age.rsquared, 3))

Age decade explains 8% of the variation in length of stay.

## Step 3 · Many predictors: binary, categorical and continuous

Heart failure is binary (0/1), admission diagnosis is categorical (`C(...)` makes one 0/1 column per group, compared with cardiac), systolic blood pressure is continuous. Each coefficient is the change in days when its predictor goes up by 1, with the others held fixed. The coefficients change from the age-only model because each one is now adjusted for the others.

In [ ]:
model_los = smf.ols("los_days ~ age_decade + heart_failure + C(admit_diagnosis_group) + systolic_bp",
                    data=df).fit()
comparison = pd.DataFrame({"Age only": model_age.params, "Four predictors": model_los.params})
print(comparison.round(4).fillna("-"))
print("R²:", round(model_age.rsquared, 3), "->", round(model_los.rsquared, 3))

In [ ]:
print("P00503, age only:      ", round(model_age.predict(patient_503).iloc[0], 2), "days")
print("P00503, four predictors:", round(model_los.predict(patient_503).iloc[0], 2), "days   (actual 11.0)")

## Step 4 · A yes/no outcome: from the odds ratio to logistic regression

The 2×2 table of heart failure and in-hospital death, as in the epidemiology refresher:

In [ ]:
table = pd.crosstab(df["heart_failure"], df["in_hospital_death"])
print(table)
a, b = table.loc[1, 1], table.loc[1, 0]   # heart failure: died, survived
c, d = table.loc[0, 1], table.loc[0, 0]   # no heart failure: died, survived
odds_ratio = (a * d) / (b * c)
print("OR:", round(odds_ratio, 2), "   log(OR):", round(np.log(odds_ratio), 3))

A logistic regression with heart failure as its only predictor. Its coefficient is log(OR), and its intercept is the log-odds without heart failure.

In [ ]:
model_hf = smf.logit("in_hospital_death ~ heart_failure", data=df).fit(disp=0)
print(model_hf.params.round(3))
print("exp(β) =", round(np.exp(model_hf.params["heart_failure"]), 2))

Risk in three steps: add up z (the log-odds) from the coefficients, odds = e<sup>z</sup>, risk = odds ÷ (1 + odds).

In [ ]:
b0, b1 = model_hf.params["Intercept"], model_hf.params["heart_failure"]
for hf in [0, 1]:
    z = b0 + b1 * hf
    odds = np.exp(z)
    print(f"Heart failure = {hf}: z = {z:.2f}, odds = {odds:.3f}, risk = {odds / (1 + odds):.3f}")
print(df.groupby("heart_failure")["in_hospital_death"].mean().round(3))

The model's risks are exactly the death rates in the two rows of the table.

## Step 5 · Age, then many predictors

A continuous predictor: the coefficient is per decade of age.

In [ ]:
model_dec = smf.logit("in_hospital_death ~ age_decade", data=df).fit(disp=0)
print(model_dec.params.round(3))
print("OR per decade:", round(np.exp(model_dec.params["age_decade"]), 2))
print(model_dec.predict(pd.DataFrame({"age_decade": [5, 8]})).round(3))

Several predictors: z is a linear combination of all of them. `np.exp` of the coefficients gives the adjusted ORs.

In [ ]:
model_death = smf.logit("in_hospital_death ~ age_decade + heart_failure + resp_rate + spo2",
                        data=df).fit(disp=0)
print(pd.DataFrame({"β": model_death.params, "OR": np.exp(model_death.params)}).round(3))

### Your turn 2

Patient P00503: eighties, no heart failure, respiratory rate 25. Fill in P00503's SpO₂, 91, and run.

In [ ]:
patient = pd.DataFrame({"age_decade": [8], "heart_failure": [0], "resp_rate": [25], "spo2": [____]})
print("Risk of in-hospital death:", round(model_death.predict(patient)[0], 3))

The same model gives every admission a risk. The ten highest:

In [ ]:
df["predicted_risk"] = model_death.predict(df)
columns = ["patient_id", "age", "heart_failure", "resp_rate", "spo2", "predicted_risk", "in_hospital_death"]
df.sort_values("predicted_risk", ascending=False).head(10)[columns].round(2)

## Key points

- Without a model the best guess is the mean. A regression line does better: intercept = the prediction at 0, slope = the change per unit.
- Each stay = mean + what the line explains + error. R² = explained ÷ total sum of squares.
- More predictors, of any type (binary, categorical, continuous), extend the same equation.
- For a yes/no outcome, logistic regression models the log-odds. With one predictor, the coefficient is log(OR).
- Risk: z = the sum of the coefficients times the patient's values; odds = e<sup>z</sup>; risk = odds ÷ (1 + odds).

## Further reading

- James, Witten, Hastie, Tibshirani and Taylor. *An Introduction to Statistical Learning with Applications in Python* (ISLP), sections 3.1–3.3 and 4.1–4.3. Free at [statlearning.com](https://www.statlearning.com).
- Enrichment notebook: scaling and significance in linear regression; the logit, odds ratios and the road to AUROC.